# Machine Learning Lab — Corrected Final Notebook

This notebook follows the **TP numbering question by question** and includes:
- all required code blocks
- beginner-friendly explanations
- troubleshooting notes
- the missing Ridge, Lasso, threshold optimization, XGBoost tuning, and final comparison sections

Run the notebook **from top to bottom**.


## Setup

In [ ]:
# Run this once in Google Colab if needed
!pip install xgboost -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from time import perf_counter

from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay, f1_score, accuracy_score
)
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)

### Setup explanation

This cell imports all libraries needed in the TP.

### Common errors
- If `ModuleNotFoundError: No module named 'xgboost'` appears, rerun the install cell.
- If a variable is not defined later, it usually means you skipped a previous cell.
- Keep `random_state=42` so your results stay reproducible.


# PART 1 — Exploring Synthetic Datasets

## Q1.1 — Display the first 5 rows of `df_reg` and comment on the value range of each feature.

In [ ]:
# Generate all datasets first, exactly as required by the TP
X_reg, y_reg = make_regression(
    n_samples=500,
    n_features=5,
    n_informative=3,
    noise=20,
    random_state=42
)

cols_reg = ['area', 'rooms', 'dist_center', 'floor', 'year']
df_reg = pd.DataFrame(X_reg, columns=cols_reg)
df_reg['price'] = y_reg

X_clf, y_clf = make_classification(
    n_samples=800,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_classes=2,
    class_sep=1.2,
    random_state=42
)

X_xgb, y_xgb = make_classification(
    n_samples=1000,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    n_classes=3,
    n_clusters_per_class=2,
    random_state=42
)

print('Shapes:', X_reg.shape, X_clf.shape, X_xgb.shape)
print('Class distribution (clf):', np.bincount(y_clf))
print('Class distribution (xgb):', np.bincount(y_xgb))

df_reg.head()

**Explanation:**  
We first generate the three synthetic datasets required in the TP, then display the first 5 rows of the regression dataframe.

The values of the features are centered around 0 and usually stay in a moderate range because the synthetic data is generated from random distributions. This is normal.

### If something goes wrong
If `df_reg` is not defined, rerun the cell above because it creates all datasets.


## Q1.2 — Compute the mean and standard deviation of the target variable `price` using `.describe()`.

In [ ]:
df_reg['price'].describe()

**Explanation:**  
The `describe()` function gives:
- mean
- standard deviation
- minimum
- maximum
- quartiles

The **mean** tells us the average of `price`, and the **standard deviation** tells us how spread out the target values are.


## Q1.3 — Plot a histogram of `y_clf`. Are both classes equally represented?

In [ ]:
plt.figure(figsize=(5,4))
plt.hist(y_clf, bins=2, edgecolor='black')
plt.title('Histogram of y_clf')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

**Explanation:**  
This histogram shows how many observations belong to class 0 and class 1.

If both bars are close in height, the classes are balanced. In this dataset they are approximately equally represented, which is good for classification.


## Q1.4 — What is the `n_informative` parameter? What is its purpose?

`n_informative` is the number of features that truly contain useful information for predicting the target.

Its purpose is to make the synthetic dataset more realistic, because in real applications not every variable is useful.


## Q1.5 — Which dataset will be used for linear regression? For classification? Briefly justify your choice.

- `X_reg, y_reg` → **Linear Regression**, because `price` is a continuous target  
- `X_clf, y_clf` → **Logistic Regression**, because the target has 2 classes  
- `X_xgb, y_xgb` → **XGBoost**, because it is a 3-class classification problem


# PART 2 — Linear Regression

## Q2.1 — What values do you get for RMSE, MAE, and R²?

In [ ]:
# Train/test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Pipeline: scaling + linear regression
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

pipe.fit(X_tr, y_tr)
y_pred = pipe.predict(X_te)

mse = mean_squared_error(y_te, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_te, y_pred)
r2 = r2_score(y_te, y_pred)

print(f'RMSE = {rmse:.2f}')
print(f'MAE  = {mae:.2f}')
print(f'R2   = {r2:.4f}')

**Explanation:**  
We train a baseline linear regression model and evaluate it with:
- **RMSE**
- **MAE**
- **R²**

A good model usually has:
- low RMSE
- low MAE
- high R²


## Q2.2 — Is the residuals plot centered around 0? What does this indicate about model quality?

In [ ]:
residuals = y_te - y_pred

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.scatter(y_pred, residuals, alpha=0.6)
plt.axhline(0, color='red')
plt.xlabel('Predicted values')
plt.ylabel('Residuals')
plt.title('Residuals vs Predictions')

plt.subplot(1,2,2)
plt.hist(residuals, bins=30, edgecolor='black')
plt.title('Residual distribution')

plt.tight_layout()
plt.show()

**Explanation:**  
Residuals are the prediction errors:

`residual = true value - predicted value`

If the residuals are spread around 0 with no strong pattern, the fit is generally good and the model is not systematically overpredicting or underpredicting.


## Q2.3 — What is the difference between MSE and RMSE? Which one is easier to interpret and why?

- **MSE** is the mean squared error  
- **RMSE** is the square root of MSE

RMSE is easier to interpret because it has the **same unit as the target variable**, while MSE is in squared units.


## Q2.4 — What is regularization? In one sentence, explain its purpose.

Regularization is a method that penalizes large coefficients in order to reduce overfitting and improve generalization.


## Q2.5 — What is the difference between Ridge (L2) and Lasso (L1)? Which one can set coefficients exactly to zero?

- **Ridge (L2)** shrinks coefficients but usually does not make them exactly zero  
- **Lasso (L1)** can force some coefficients to become exactly zero

So **Lasso** is the one that can set coefficients exactly to zero.


In [ ]:
# Ridge regularization
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge())
])

param_grid_ridge = {'model__alpha': [0.01, 0.1, 1, 10, 100]}
gs_ridge = GridSearchCV(ridge_pipe, param_grid_ridge, cv=5, scoring='neg_mean_squared_error')
gs_ridge.fit(X_tr, y_tr)

print('Ridge best alpha:', gs_ridge.best_params_)

# Lasso regularization
lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(max_iter=5000))
])

param_grid_lasso = {'model__alpha': [0.001, 0.01, 0.1, 1, 10]}
gs_lasso = GridSearchCV(lasso_pipe, param_grid_lasso, cv=5, scoring='neg_mean_squared_error')
gs_lasso.fit(X_tr, y_tr)

print('Lasso best alpha:', gs_lasso.best_params_)

lasso_best = gs_lasso.best_estimator_.named_steps['model']
print('Lasso coefficients:', lasso_best.coef_)

**Explanation:**  
This cell trains **Ridge** and **Lasso** and uses `GridSearchCV` to find the best value of `alpha`.

### If something goes wrong
If this cell takes some time, that is normal because cross-validation trains several models.


## Q2.6 — Run Ridge with `alpha=0.01` then `alpha=100`. How do the coefficients change as alpha increases?

In [ ]:
ridge_small = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=0.01))
])
ridge_large = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=100))
])

ridge_small.fit(X_tr, y_tr)
ridge_large.fit(X_tr, y_tr)

coef_small = ridge_small.named_steps['model'].coef_
coef_large = ridge_large.named_steps['model'].coef_

print("Ridge coefficients with alpha=0.01:")
print(coef_small)
print("\nRidge coefficients with alpha=100:")
print(coef_large)

**Explanation:**  
As `alpha` increases, the penalty becomes stronger, so the coefficients usually become smaller in magnitude.


## Q2.7 — After training Lasso, display `lasso_best.coef_`. How many coefficients are zero? What does this mean?

In [ ]:
num_zero = np.sum(lasso_best.coef_ == 0)
print("Number of zero coefficients in Lasso:", num_zero)

**Explanation:**  
If some Lasso coefficients are exactly zero, it means the model removed those features. This is why Lasso can work as a basic feature selection method.


## Q2.8 — What optimal alpha was found by GridSearchCV for Ridge? And for Lasso? Do performances improve vs plain linear regression?

In [ ]:
y_pred_ridge = gs_ridge.best_estimator_.predict(X_te)
y_pred_lasso = gs_lasso.best_estimator_.predict(X_te)

print("Baseline RMSE:", round(np.sqrt(mean_squared_error(y_te, y_pred)), 2))
print("Best Ridge RMSE:", round(np.sqrt(mean_squared_error(y_te, y_pred_ridge)), 2))
print("Best Lasso RMSE:", round(np.sqrt(mean_squared_error(y_te, y_pred_lasso)), 2))

**Explanation:**  
Compare the best Ridge and Lasso RMSE with the baseline RMSE. If the RMSE becomes smaller, then performance improved.


## Q2.9 — In which cases would you prefer Lasso over Ridge? Give a concrete example.

We prefer **Lasso** when we believe many features are not useful and we want the model to remove them.

**Example:** if we have many customer variables but only a few are truly important, Lasso helps keep only the useful ones.


# PART 3 — Logistic Regression

## Q3.1 — What accuracy and F1-score do you obtain? Is the model satisfactory?

In [ ]:
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_clf, y_clf, test_size=0.2, stratify=y_clf, random_state=42
)

log_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

log_pipe.fit(X_tr2, y_tr2)

y_pred2 = log_pipe.predict(X_te2)
y_proba = log_pipe.predict_proba(X_te2)[:, 1]

print(classification_report(y_te2, y_pred2, target_names=['Ham', 'Spam']))
print("Accuracy:", round(accuracy_score(y_te2, y_pred2), 4))
print("F1-score:", round(f1_score(y_te2, y_pred2), 4))

**Explanation:**  
Logistic regression predicts probabilities for class membership, then converts them into class labels.

We evaluate the model with:
- classification report
- accuracy
- F1-score


## Q3.2 — Look at the confusion matrix. How many spams were correctly detected? How many legitimate emails were wrongly classified as spam?

In [ ]:
cm = confusion_matrix(y_te2, y_pred2)
ConfusionMatrixDisplay(cm, display_labels=['Ham', 'Spam']).plot()
plt.title('Confusion Matrix')
plt.show()

print("Confusion matrix:\n", cm)

**Explanation:**  
The confusion matrix shows:
- correct predictions
- incorrect predictions

To answer the question:
- spams correctly detected = true positives for Spam
- legitimate emails wrongly classified as spam = false positives


## Q3.3 — What is AUC-ROC? Is a value close to 1 good or bad?

In [ ]:
fpr, tpr, _ = roc_curve(y_te2, y_proba)
auc = roc_auc_score(y_te2, y_proba)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0,1], [0,1], '--', color='gray')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curve')
plt.legend()
plt.show()

print("AUC-ROC:", round(auc, 4))

**Explanation:**  
AUC-ROC measures how well the model separates the two classes over all thresholds.

A value close to **1** is very good.


## Q3.4 — Change the parameter `C=0.01` (strong regularization). Does accuracy increase or decrease? Why?

In [ ]:
log_pipe_strong = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000, C=0.01, random_state=42))
])

log_pipe_strong.fit(X_tr2, y_tr2)
y_pred2_strong = log_pipe_strong.predict(X_te2)

print("Default accuracy:", round(accuracy_score(y_te2, y_pred2), 4))
print("Accuracy with C=0.01:", round(accuracy_score(y_te2, y_pred2_strong), 4))

**Explanation:**  
A smaller `C` means stronger regularization.

If accuracy decreases, the model became too constrained. If it improves slightly, the simpler model generalized better.


## Q3.5 — What is the decision threshold (0.5 by default)? What happens if you change it to 0.3?

The decision threshold is the probability cutoff used to assign class 1.

At threshold 0.5:
- if probability ≥ 0.5 → class 1

At threshold 0.3:
- more observations are classified as class 1
- recall usually increases
- precision may decrease


## Q3.6 — Based on the F1 vs threshold curve, which threshold would you choose? Justify your answer.

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)
f1_scores = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    f1_scores.append(f1_score(y_te2, y_pred_t))

best_t = thresholds[np.argmax(f1_scores)]
print(f'Optimal threshold (F1): {best_t:.2f}')

plt.figure(figsize=(6,4))
plt.plot(thresholds, f1_scores, marker='o')
plt.axvline(best_t, color='red', linestyle='--', label=f'Optimal = {best_t:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1-score')
plt.title('F1-score by threshold')
plt.legend()
plt.show()

**Explanation:**  
We choose the threshold that gives the **highest F1-score**, because it gives the best balance between precision and recall.


# PART 4 — XGBoost – Gradient Boosting

## Q4.1 — What accuracy do you get with XGBoost? Compare it to logistic regression (Part 3).

In [ ]:
X_tr3, X_te3, y_tr3, y_te3 = train_test_split(
    X_xgb, y_xgb, test_size=0.2, stratify=y_xgb, random_state=42
)

xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)

xgb_clf.fit(
    X_tr3, y_tr3,
    eval_set=[(X_te3, y_te3)],
    verbose=False
)

y_pred3 = xgb_clf.predict(X_te3)

print("Accuracy:", round(accuracy_score(y_te3, y_pred3), 4))
print(classification_report(y_te3, y_pred3, target_names=['Species A','Species B','Species C']))

**Explanation:**  
XGBoost is an ensemble model based on boosting. It builds trees sequentially and each tree tries to correct previous errors.


## Q4.2 — What is boosting? In one or two sentences, explain the main idea.

Boosting is an ensemble technique where weak learners are trained one after another, and each new learner focuses on correcting the mistakes made by the previous ones.


## Q4.3 — Look at the convergence curve (`mlogloss`). From which iteration does the loss stabilize?

In [ ]:
results = xgb_clf.evals_result()

plt.figure(figsize=(6,4))
plt.plot(results['validation_0']['mlogloss'], label='Test')
plt.xlabel('Iteration')
plt.ylabel('mlogloss')
plt.title('XGBoost Convergence')
plt.legend()
plt.show()

**Explanation:**  
Look at where the curve starts becoming almost flat. That approximate point is where the loss stabilizes.


## Q4.4 — What are the 3 most important features according to the importance plot?

In [ ]:
feat_imp = pd.Series(
    xgb_clf.feature_importances_,
    index=[f'feat_{i}' for i in range(X_xgb.shape[1])]
)

feat_imp.sort_values().plot(kind='barh', figsize=(8,5), title='Feature Importance (XGBoost)')
plt.tight_layout()
plt.show()

print("Top 3 most important features:")
print(feat_imp.sort_values(ascending=False).head(3))

**Explanation:**  
The top 3 printed features are the most influential variables for the XGBoost model.


## Q4.5 — Try setting `learning_rate=0.5`. Is the accuracy better or worse?

In [ ]:
xgb_fast = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)

xgb_fast.fit(X_tr3, y_tr3, verbose=False)
y_pred_fast = xgb_fast.predict(X_te3)

print("Accuracy with learning_rate=0.1:", round(accuracy_score(y_te3, y_pred3), 4))
print("Accuracy with learning_rate=0.5:", round(accuracy_score(y_te3, y_pred_fast), 4))

**Explanation:**  
A higher learning rate can make training less stable, so the accuracy may get worse.


## Q4.6 — After RandomizedSearchCV, what are the best hyperparameters found? Does the score improve?

In [ ]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

rs = RandomizedSearchCV(
    xgb.XGBClassifier(eval_metric='mlogloss', random_state=42),
    param_dist,
    n_iter=20,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

rs.fit(X_tr3, y_tr3)

print('Best parameters:', rs.best_params_)
print('Best CV score:', round(rs.best_score_, 4))

**Explanation:**  
This random search tests many parameter combinations and returns the best ones found.


# PART 5 — Comparison & Wrap-up

## Q5.1 — Fill in the summary table below with the metrics recorded in each part.

In [ ]:
models = {
    'Logistic Reg.': Pipeline([
        ('sc', StandardScaler()),
        ('m', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200,
        max_depth=4,
        eval_metric='logloss',
        random_state=42
    ),
}

results_comp = {}

for name, model in models.items():
    t0 = perf_counter()
    model.fit(X_tr2, y_tr2)
    fit_time = perf_counter() - t0

    yp = model.predict(X_te2)
    yprb = model.predict_proba(X_te2)[:, 1]

    results_comp[name] = {
        'Accuracy': accuracy_score(y_te2, yp),
        'F1': f1_score(y_te2, yp),
        'AUC-ROC': roc_auc_score(y_te2, yprb),
        'Fit time (s)': fit_time
    }

df_results = pd.DataFrame(results_comp).T
print(df_results.to_string(float_format='{:.4f}'.format))

In [ ]:
df_results.plot(kind='bar', figsize=(10,5), edgecolor='black')
plt.title('Model Comparison')
plt.xticks(rotation=0)
plt.ylim(0.0, 1.05)
plt.tight_layout()
plt.show()

**Explanation:**  
Here we compare Logistic Regression, Random Forest, and XGBoost on the **same binary dataset**, which makes the comparison fair.


## Q5.2 — Which model achieved the best performance? Was this expected?

Use the comparison table above to identify the best-performing model.

In many tabular-data problems, tree-based models such as Random Forest and XGBoost often perform very well, so this is often expected.


## Q5.3 — Which model is the easiest to interpret? Give one concrete advantage of this interpretability.

The easiest model to interpret is **Logistic Regression**.

A concrete advantage is that we can inspect the coefficients directly and understand how each feature influences the prediction.


## Q5.4 — If you had to choose a single model for a real-world use case, which one would you pick and why?

A reasonable answer is:
- choose **Logistic Regression** if simplicity and interpretability are most important
- choose **XGBoost** if predictive performance is the top priority
- choose **Random Forest** if you want a strong model that is usually easier to tune than XGBoost


## Final check

This corrected version now includes:
- all 3 datasets
- Part 1 questions Q1.1 to Q1.5
- baseline linear regression
- Ridge and Lasso
- GridSearchCV
- logistic regression
- confusion matrix
- ROC curve
- threshold optimization
- XGBoost baseline
- convergence curve
- feature importance
- learning-rate experiment
- RandomizedSearchCV
- final comparison with Random Forest

This is the corrected TP-complete notebook version.
